# IMR exploratory analysis

Overturn rates in the California DMHC Independent Medical Review data, by treatment, diagnosis, year and case type, plus the three PAIR policy groups.

Data: `make data` (never committed). Schema and leakage rules: `docs/data/imr-schema.md`. Outputs are stripped before commit (`make strip-notebooks`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pair.score.imr_data import load_imr_features
from pair.score.imr_mapping import load_imr_mapping, select_rows

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
frame = load_imr_features(REPO / "data/raw/imr/imr_determinations.csv")
print(frame.shape, f"overall overturn rate: {frame.label.mean():.3f}")

In [ ]:
def overturn_by(column: str, min_n: int = 100) -> pd.DataFrame:
    """Overturn rate and case count per level of ``column``, for levels with at least ``min_n`` cases."""
    table = frame.groupby(column, observed=True)["label"].agg(n="size", overturn_rate="mean")
    return table[table.n >= min_n].sort_values("n", ascending=False)


overturn_by("TreatmentCategory").head(15)

In [ ]:
overturn_by("DiagnosisCategory").head(15)

In [ ]:
pd.concat({"Type": overturn_by("Type", 0), "IMRType": overturn_by("IMRType", 0)})

## By year

The planned split trains on 2022 and earlier, calibrates on 2023 and tests on 2024 onward. 2026 is a partial year.

In [ ]:
by_year = frame.groupby("report_year")["label"].agg(n="size", overturn_rate="mean")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(by_year.index, by_year.overturn_rate, marker="o")
ax.set(xlabel="Report year", ylabel="Overturn rate", title="IMR overturn rate by year")
for year in (2022.5, 2023.5):
    ax.axvline(year, linestyle="--", linewidth=1)
plt.tight_layout()
by_year.T

## PAIR policy groups

Rows matched by `policies/_imr-mapping.yaml`. These counts set how wide each policy's base-rate interval can be.

In [ ]:
mapping = load_imr_mapping(REPO / "policies/_imr-mapping.yaml")
rows = []
for policy_id, policy_mapping in mapping.items():
    group = frame[select_rows(frame, policy_mapping)]
    rows.append({
        "policy": policy_id,
        "n": len(group),
        "overturn_rate": round(group.label.mean(), 3),
        "n_2024_plus": int((group.report_year >= 2024).sum()),
        "years": f"{group.report_year.min()}-{group.report_year.max()}",
    })
pd.DataFrame(rows).set_index("policy")